# Activity 4: Mentor Matching & Intervention Recommendations

## Objective
Translate student insights into real mentoring actions using:
- Student Scores
- Student Clusters
- Mentor Dataset

The system assigns appropriate mentors based on:
- Student needs
- Mentor expertise
- Mentor availability
- Capacity constraints


In [3]:
import pandas as pd
# Load student datasets
students_scored = pd.read_csv("students_scored.csv")
clusters = pd.read_csv("activity3_student_clusters.csv")

# Load mentor dataset
mentors = pd.read_csv("mentor.csv")

# Merge students with cluster info
students = students_scored.merge(
    clusters[['student_id', 'Cluster']],
    on='student_id',
    how='left'
)

students.head()


,student_id,age,program,semester,gpa,attendance,assignments_completion,stress_level,sleep_hours,mental_wellbeing,...,engagement_score,APS,sleep_score,stress_score,WWS,PTMS,CRS,SRI,Category,Cluster
0,S001,24,B.Tech,7,6.56,91,89,4,8.7,8,...,84,77.90,76.0,60,69.6,33.3,22.0,52.930,Yellow,1
1,S002,21,B.Sc,7,8.50,91,88,8,8.8,3,...,43,87.40,74.0,20,52.4,27.1,12.2,47.790,Red,2
2,S003,22,B.Sc,6,8.06,83,67,4,8.6,6,...,75,78.60,78.0,60,70.8,39.5,19.0,53.930,Yellow,1
3,S004,24,B.Tech,7,9.26,71,89,4,5.9,4,...,45,85.40,68.0,60,64.8,31.0,12.6,51.170,Yellow,2
4,S005,20,B.Tech,4,7.27,98,50,8,4.1,8,...,99,75.75,32.0,20,27.2,38.3,23.4,43.035,Red,1


## Mentor Matching Logic

Cluster → Mentor Type Mapping:

- High Risk → Wellness Mentor
- Academic Risk → Academic Mentor
- Career Confused → Career Mentor
- Stable → Academic Growth Mentor


###  Decision Function

In [4]:
def decide_mentor(cluster):
    if cluster == "High Risk":
        return "Wellness", "Immediate Counseling & Weekly Monitoring"
    elif cluster == "Academic Risk":
        return "Academic", "Subject Support & Study Plan"
    elif cluster == "Career Confused":
        return "Career", "Resume Building & Career Guidance"
    else:
        return "Academic", "Growth Mentorship"


### Matching Function

In [5]:
def match_mentor(row):
    mentor_type, intervention = decide_mentor(row['Cluster'])
    
    eligible = mentors[
        (mentors['expertise_area'] == mentor_type) &
        (mentors['availability'] == "Available") &
        (mentors['current_load'] < mentors['max_students'])
    ]
    
    if len(eligible) > 0:
        selected = eligible.iloc[0]
        
        # Update mentor load
        mentors.loc[
            mentors['mentor_id'] == selected['mentor_id'], 
            'current_load'
        ] += 1
        
        return pd.Series([
            selected['mentor_name'],
            mentor_type,
            intervention
        ])
    else:
        return pd.Series([
            "No Mentor Available",
            mentor_type,
            intervention
        ])


### Apply Matching

In [6]:
students[['Assigned_Mentor', 
          'Mentor_Type', 
          'Intervention']] = students.apply(match_mentor, axis=1)

students.head()


,student_id,age,program,semester,gpa,attendance,assignments_completion,stress_level,sleep_hours,mental_wellbeing,...,stress_score,WWS,PTMS,CRS,SRI,Category,Cluster,Assigned_Mentor,Mentor_Type,Intervention
0,S001,24,B.Tech,7,6.56,91,89,4,8.7,8,...,60,69.6,33.3,22.0,52.930,Yellow,1,Dr. Sharma,Academic,Growth Mentorship
1,S002,21,B.Sc,7,8.50,91,88,8,8.8,3,...,20,52.4,27.1,12.2,47.790,Red,2,Dr. Sharma,Academic,Growth Mentorship
2,S003,22,B.Sc,6,8.06,83,67,4,8.6,6,...,60,70.8,39.5,19.0,53.930,Yellow,1,Dr. Sharma,Academic,Growth Mentorship
3,S004,24,B.Tech,7,9.26,71,89,4,5.9,4,...,60,64.8,31.0,12.6,51.170,Yellow,2,Dr. Sharma,Academic,Growth Mentorship
4,S005,20,B.Tech,4,7.27,98,50,8,4.1,8,...,20,27.2,38.3,23.4,43.035,Red,1,Prof. Rao,Academic,Growth Mentorship


## High-Risk Alert Simulation

Students in the "High Risk" cluster are flagged for immediate intervention.


### Alert System

In [8]:
students['Alert'] = students['Cluster'].apply(
    lambda x: "YES " if x == "High Risk" else "NO"
)

# Display high-risk students
high_risk_students = students[students['Cluster'] == "High Risk"]

for _, row in high_risk_students.iterrows():
    print(f"ALERT : Student {row['student_id']} requires immediate intervention!")

students.head()


,student_id,age,program,semester,gpa,attendance,assignments_completion,stress_level,sleep_hours,mental_wellbeing,...,WWS,PTMS,CRS,SRI,Category,Cluster,Assigned_Mentor,Mentor_Type,Intervention,Alert
0,S001,24,B.Tech,7,6.56,91,89,4,8.7,8,...,69.6,33.3,22.0,52.930,Yellow,1,Dr. Sharma,Academic,Growth Mentorship,NO
1,S002,21,B.Sc,7,8.50,91,88,8,8.8,3,...,52.4,27.1,12.2,47.790,Red,2,Dr. Sharma,Academic,Growth Mentorship,NO
2,S003,22,B.Sc,6,8.06,83,67,4,8.6,6,...,70.8,39.5,19.0,53.930,Yellow,1,Dr. Sharma,Academic,Growth Mentorship,NO
3,S004,24,B.Tech,7,9.26,71,89,4,5.9,4,...,64.8,31.0,12.6,51.170,Yellow,2,Dr. Sharma,Academic,Growth Mentorship,NO
4,S005,20,B.Tech,4,7.27,98,50,8,4.1,8,...,27.2,38.3,23.4,43.035,Red,1,Prof. Rao,Academic,Growth Mentorship,NO


### Save Final Output

In [9]:
students.to_csv("final_mentor_recommendations.csv", index=False)

print("Final recommendation file saved successfully.")


Final recommendation file saved successfully.
